## 1. **`pd.cut()`:**

- `pd.cut()` in pandas is used to **bin continuous data into discrete intervals**. 

- It’s helpful when you want to transform numerical values into categorical ranges or "bins."

- It splits the values of a continuous variable into intervals (bins), and assigns each value to one of those bins. 

**Examples:**  
* Creating age groups (e.g., 0–18, 18–35, etc.)
* Discretizing continuous features before modeling
* Making data easier to interpret or visualize

```python
pd.cut(x, bins, right=True, labels=None, include_lowest=False)
```

| Parameter        | Description                                                   |
| ---------------- | ------------------------------------------------------------- |
| `x`              | Array-like or Series to be binned                             |
| `bins`           | Number of bins, or a list of bin edges                        |
| `right`          | If True (default), bins include the right edge (e.g., (a, b]) |
| `labels`         | Optional custom labels for the bins                           |
| `include_lowest` | If True, the first interval includes the leftmost edge        |

### **Example 1: Binning into equal-width intervals:**

In [1]:
import pandas as pd

ages = [15, 25, 45, 60, 75]
bins = [0, 18, 35, 65, 100]
age_groups = pd.cut(ages, bins)

print(age_groups)

[(0, 18], (18, 35], (35, 65], (35, 65], (65, 100]]
Categories (4, interval[int64, right]): [(0, 18] < (18, 35] < (35, 65] < (65, 100]]


### **Example 2: Using custom labels:**

In [2]:
labels = ['Teen', 'Young Adult', 'Adult', 'Senior']
age_groups = pd.cut(ages, bins, labels=labels)

print(age_groups)

['Teen', 'Young Adult', 'Adult', 'Adult', 'Senior']
Categories (4, object): ['Teen' < 'Young Adult' < 'Adult' < 'Senior']


### **Example 3: Automatically bin values into equal-width intervals:**

In [3]:
data = pd.Series([10, 20, 30, 40, 50])
binned = pd.cut(data, bins=3)

print(binned)

0      (9.96, 23.333]
1      (9.96, 23.333]
2    (23.333, 36.667]
3      (36.667, 50.0]
4      (36.667, 50.0]
dtype: category
Categories (3, interval[float64, right]): [(9.96, 23.333] < (23.333, 36.667] < (36.667, 50.0]]


**This splits the data into 3 equal-width bins.**

### **Example 4: Include the lowest value:**

In [4]:
pd.cut([0, 1, 2, 3, 4], bins=[0, 2, 4], include_lowest=True)

[(-0.001, 2.0], (-0.001, 2.0], (-0.001, 2.0], (2.0, 4.0], (2.0, 4.0]]
Categories (2, interval[float64, right]): [(-0.001, 2.0] < (2.0, 4.0]]

This ensures the value `0` is included in the first bin.

### **Use Cases:**

* Grouping ages, income ranges, or temperature values
* Feature engineering in ML (discretizing continuous variables)
* Summarizing data by category (e.g., histogram-style buckets)

----
----
---

## 2. **What is `pd.qcut()`:**

- `pandas.qcut()` is used to **bin continuous data into quantile-based buckets**. 

- Unlike `pd.cut()` which splits based on equal-width intervals or fixed bins, `qcut()` splits data such that **each bin has (approximately) the same number of data points**.

**It’s useful when:**  

  * We want to divide our data into `quartiles`, `quintiles`, `deciles`, etc.
  * You're doing feature engineering and need evenly populated bins.
  * You're analyzing customer spending, income, or scores and want to create segments with equal size.

> If our data has **duplicate values at bin edges**, `qcut()` might fail; use `duplicates='drop'` to skip problematic bins.

> It’s ideal when data is skewed and `pd.cut()` would create uneven groups.

> If we want exact quantile values, use `df['col'].quantile([0.25, 0.5, 0.75])`.


```python
pd.qcut(x, q, labels=False, precision=3, duplicates='raise')
```

| Argument     | Description                                                           |
| ------------ | --------------------------------------------------------------------- |
| `x`          | 1D array-like or Series to be binned                                  |
| `q`          | Number of quantiles or list of quantile edges (e.g., 4 for quartiles) |
| `labels`     | Assign custom labels or `False` to return bin numbers                 |
| `precision`  | Decimal places of bin edges                                           |
| `duplicates` | If 'drop', drops non-unique bins                                      |

**Example 1: Divide into Quartiles:**

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Score': np.random.randint(1, 101, 1000)  # 1000 random scores from 1 to 100
})
df.head(4)

,Score
0,6
1,55
2,29
3,81


In [3]:
df['Quartile'] = pd.qcut(df['Score'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(df.head())

   Score Quartile
0      6       Q1
1     55       Q3
2     29       Q2
3     81       Q4
4     30       Q2


This creates 4 bins (quartiles) each with \~250 data points.

**Example 2: Get bin numbers instead of labels:**

In [4]:
df['Quantile_Num'] = pd.qcut(df['Score'], q=4, labels=False)
df.head(4)

,Score,Quartile,Quantile_Num
0,6,Q1,0
1,55,Q3,2
2,29,Q2,1
3,81,Q4,3


**Example 3: Custom quantiles (e.g., deciles):**

In [5]:
df['Decile'] = pd.qcut(df['Score'], q=10, labels=False)
df.head(10)

,Score,Quartile,Quantile_Num,Decile
0,6,Q1,0,0
1,55,Q3,2,5
2,29,Q2,1,2
3,81,Q4,3,7
4,30,Q2,1,2
5,46,Q2,1,4
6,8,Q1,0,0
7,45,Q2,1,4
8,74,Q3,2,7
9,14,Q1,0,1


**Example 4: Using `qcut()` on a large DataFrame column:**

In [6]:
# Simulate a large DataFrame
df_big = pd.DataFrame({
    'User_ID': np.arange(1, 1_000_001),
    'Spending': np.random.exponential(scale=1000, size=1_000_000)  # skewed distribution
})
df_big.head()

,User_ID,Spending
0,1,600.545734
1,2,887.126996
2,3,5760.583726
3,4,293.885015
4,5,697.800470


In [7]:
df_big.shape

(1000000, 2)

In [ ]:
# Segment into 5 equal-sized bins
df_big['Spending_Segment'] = pd.qcut(df_big['Spending'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
df_big.head()

,User_ID,Spending,Spending_Segment
0,1,600.545734,Medium
1,2,887.126996,Medium
2,3,5760.583726,Very High
3,4,293.885015,Low
4,5,697.800470,Medium


In [10]:
# Example: Count how many users in each spending group
print(df_big['Spending_Segment'].value_counts())

Spending_Segment
Very Low     200000
Low          200000
Medium       200000
High         200000
Very High    200000
Name: count, dtype: int64
